In [7]:
import dataclasses
from dataclasses import dataclass, field, fields

class ProtectedString(str):
    pass

@dataclass
class MetadataAwareBase:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        annotated_fields = cls.__annotations__
        metadata_to_add = {'sensitive': True}

        for name, field_type in annotated_fields.items():
            if field_type is ProtectedString:
                current_value = getattr(cls, name, None)
                if isinstance(current_value, dataclasses.Field):
                    existing_metadata = current_value.metadata or {}
                    new_metadata = {**existing_metadata, **metadata_to_add}
                    new_field = dataclasses.replace(current_value, metadata=new_metadata)
                else:
                    new_field = field(default=current_value, metadata=metadata_to_add)

                setattr(cls, name, new_field)

# @dataclass
class UserCredentials(MetadataAwareBase):
    user_id: int  # This field will be ignored by our logic.
    username: ProtectedString  # This field should get the metadata.
    api_key: ProtectedString = field(default="default_key")
    is_active: bool = True

TypeError: replace() should be called on dataclass instances

In [ ]:

# 4. Demonstrate that the metadata was added correctly.
if __name__ == "__main__":
    print(f"--- Analyzing fields for the '{UserCredentials.__name__}' dataclass ---")

    # The `dataclasses.fields()` utility function inspects the final dataclass.
    all_fields = fields(UserCredentials)

    for f in all_fields:
        print(f"\nField: '{f.name}'")
        print(f"  - Type: {f.type}")
        # The metadata attribute is a mappingproxy, so we convert it to a dict for display.
        print(f"  - Metadata: {dict(f.metadata)}")
        print(f"  - Has 'sensitive' metadata? {'sensitive' in f.metadata}")

    print("\n--- Creating an instance of UserCredentials ---")
    user = UserCredentials(user_id=123, username="testuser")
    print(user)
    print(f"Instance's API key: {user.api_key}")

--- Analyzing fields for the 'UserCredentials' dataclass ---

--- Creating an instance of UserCredentials ---


TypeError: MetadataAwareBase.__init__() got an unexpected keyword argument 'user_id'